<a href="https://colab.research.google.com/github/marcohinojosag/IA_MKTR/blob/branch-y-histograma/cod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. Importación de librerias y carga del dataset
Quien necesita descargar en 2025, las APIs son el futuro 🗣🗣🗣

In [9]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
import pandas as pd
import numpy as np
from kagglehub import KaggleDatasetAdapter
import re, numpy as np, pandas as pd
from scipy.stats import skew as sp_skew, kurtosis as sp_kurtosis
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

In [2]:
# Sets path
test_path = "aps_failure_test_set_processed_8bit.csv"
train_path = "aps_failure_training_set_processed_8bit.csv"

# Load the latest version
test = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "uciml/aps-failure-at-scania-trucks-data-set",
  test_path,
)

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "uciml/aps-failure-at-scania-trucks-data-set",
  train_path,
)

100%|██████████| 2.99M/2.99M [00:00<00:00, 127MB/s]

Extracting zip of aps_failure_test_set_processed_8bit.csv...


100%|██████████| 11.2M/11.2M [00:00<00:00, 68.6MB/s]

Extracting zip of aps_failure_training_set_processed_8bit.csv...


# 1. Analisis del dataset


In [3]:
# Muestra de los datos
print(df.head(5))

      class    aa_000    ab_000    ac_000    ad_000    ae_000    af_000  \
0 -0.992188  0.117188 -0.289062  0.992188 -0.007812 -0.046875 -0.054688   
1 -0.992188 -0.179688 -0.289062 -0.468750 -0.007812 -0.046875 -0.054688   
2 -0.992188 -0.125000 -0.289062 -0.468750 -0.007812 -0.046875 -0.054688   
3 -0.992188 -0.406250 -0.289062 -0.468750 -0.007812 -0.046875 -0.007812   
4 -0.992188  0.007812 -0.289062 -0.468750 -0.007812 -0.046875 -0.054688   

     ag_000   ag_001    ag_002  ...    ee_002    ee_003    ee_004    ee_005  \
0 -0.007812 -0.03125 -0.054688  ...  0.687500  0.515625  0.234375  0.070312   
1 -0.007812 -0.03125 -0.054688  ... -0.023438 -0.062500 -0.132812 -0.132812   
2 -0.007812 -0.03125 -0.054688  ... -0.140625 -0.093750 -0.015625  0.015625   
3 -0.007812 -0.03125 -0.054688  ... -0.382812 -0.382812 -0.375000 -0.351562   
4 -0.007812 -0.03125 -0.054688  ...  0.156250  0.031250 -0.031250 -0.039062   

     ee_006    ee_007    ee_008    ee_009    ef_000    eg_000  
0  0.00781

In [7]:
# Tamaño del dataset
print(df.shape)

(60000, 171)


Como que 60k filas 💀, que diga...

Descripcion de las columnas:
- clase: "neg" indica fallo en el sistema APS, "pos" todo en orden. (se renombra por palabra reservada)
- aa_000, ab_000, ac_000, ..., eg_000: atributos de entrada, sin un significado en particular (anonimos)

In [5]:
# Columnas del dataset
df.rename(columns={'class': 'clase'}, inplace=True)
df.columns

Index(['clase', 'aa_000', 'ab_000', 'ac_000', 'ad_000', 'ae_000', 'af_000',
       'ag_000', 'ag_001', 'ag_002',
       ...
       'ee_002', 'ee_003', 'ee_004', 'ee_005', 'ee_006', 'ee_007', 'ee_008',
       'ee_009', 'ef_000', 'eg_000'],
      dtype='object', length=171)

In [10]:
# Cantidad de elementos vacios por cada columna
cant_vacios = (df == "na").sum()
cant_vacios = cant_vacios[cant_vacios > 0]
print(cant_vacios)

Series([], dtype: int64)


In [11]:
# Cantidad de positivos y negativos del dataset
cant_neg = len(df[df.clase == 'neg'])
cant_pos = len(df[df.clase == 'pos'])
print(f"Negativos: {cant_neg}, Positivos: {cant_pos}")

Negativos: 0, Positivos: 0


In [16]:
# =========================
# 1) Librerías y parámetros
# =========================
import re
import numpy as np
import pandas as pd

CSV_PATH   = "tu_dataset.csv"   # <-- cámbialo
TARGET_COL = "class"            # cambia si tu etiqueta se llama distinto
MIN_BINS_FOR_HIST = 3           # mínimo de columnas por base para considerarlo histograma

# Abreviaturas :
ABBR = {
    "mean": "med",    # media
    "var":  "var",    # varianza
    "skew": "asi",    # asimetría
    "kurt": "cur",    # curtosis (Pearson, no-excesiva)
}

# Patrón base_### (ej.: aaa_000)
SUFFIX_RE = re.compile(r"^(?P<base>[A-Za-z]+)_(?P<bin>\d{3})$")


# ==========================
# 2) Detección de histogramas
# ==========================
def infer_histogram_groups(columns, target_col=TARGET_COL, min_bins=MIN_BINS_FOR_HIST):
    """
    Devuelve:
      - hist_groups: dict { base: [cols ordenadas por bin] }
      - non_hist_cols: columnas que no pertenecen a histogramas
    """
    buckets = {}
    non_hist_cols = []

    for col in columns:
        if col == target_col:
            continue
        m = SUFFIX_RE.match(col)
        if not m:
            non_hist_cols.append(col)
            continue
        base = m.group("base")
        bin_id = int(m.group("bin"))
        buckets.setdefault(base, []).append((bin_id, col))

    hist_groups = {}
    for base, pairs in buckets.items():
        pairs_sorted = sorted(pairs, key=lambda x: x[0])
        cols_sorted = [c for _, c in pairs_sorted]
        if len(cols_sorted) >= min_bins:
            hist_groups[base] = cols_sorted
        else:
            non_hist_cols.extend(cols_sorted)

    return hist_groups, non_hist_cols




def summarize_histogram_matrix_robust(mat, laplace_alpha=0.0, use_float=np.float64):
    """
    mat: array (n_rows x n_bins) con bins en 0..255 (u otra escala de 8 bits).
    Convierte cada fila a PMF SOLO para calcular momentos (no modifica tu DF).
    Devuelve dict con mean, var, skew, kurt (kurtosis de Pearson, no-excesiva).
    Filas sin masa válida => NaN en todos los estadísticos.
    """
    M = np.array(mat, dtype=use_float, copy=True)

    totals = np.nansum(M, axis=1, keepdims=True)
    all_nan_row = np.all(~np.isfinite(M), axis=1, keepdims=True)
    invalid = (totals <= 0) | ~np.isfinite(totals) | all_nan_row

    if laplace_alpha > 0:
        M = np.where(np.isfinite(M), M + laplace_alpha, laplace_alpha)

    totals = np.nansum(M, axis=1, keepdims=True)
    invalid = invalid | (totals <= 0) | ~np.isfinite(totals)

    totals_safe = np.where(invalid, np.nan, totals)
    p = M / totals_safe  # PMF por fila (solo para los momentos)

    n_bins = M.shape[1]
    x = np.arange(n_bins, dtype=use_float).reshape(1, -1)

    mu = np.nansum(p * x, axis=1, keepdims=True)
    xc = x - mu
    m2 = np.nansum(p * (xc**2), axis=1, keepdims=True)
    m3 = np.nansum(p * (xc**3), axis=1, keepdims=True)
    m4 = np.nansum(p * (xc**4), axis=1, keepdims=True)

    var = m2.squeeze(1)
    std = np.sqrt(var)

    with np.errstate(divide='ignore', invalid='ignore'):
        skew = (m3.squeeze(1)) / (std**3)
        kurt = (m4.squeeze(1)) / (var**2)

    mu   = mu.squeeze(1)
    skew = np.where(invalid.squeeze(1), np.nan, skew)
    kurt = np.where(invalid.squeeze(1), np.nan, kurt)

    return {"mean": mu, "var": var, "skew": skew, "kurt": kurt}



def three_letter_base(base):
    """Primeras 3 letras del base en minúsculas (o lo que haya si <3)."""
    return base[:3].lower()


# ===========================
# 4) Pipeline principal (E2E)
# ===========================


# Convertir a numérico cuando se pueda
for c in df.columns:
    if c != TARGET_COL:
        df[c] = pd.to_numeric(df[c], errors="ignore")

# 4.2 Detectar histogramas
hist_groups, non_hist_cols = infer_histogram_groups(df.columns)

print("Histogramas detectados:")
for base, cols in hist_groups.items():
    print(f"  {base}: {len(cols)} bins -> {cols[:5]}{' ...' if len(cols) > 5 else ''}")
print("\nColumnas NO histogramas (muestra):", non_hist_cols[:10])

# 4.3 Construir DataFrame de salida con:
#     - class (si existe)
#     - columnas NO histogramas (se conservan)
out_cols = []
if TARGET_COL in df.columns:
    out_cols.append(TARGET_COL)
out_cols += non_hist_cols

df_out = df[out_cols].copy()

# Añadir features por cada histograma detectado
for base, cols in hist_groups.items():
    X = df[cols].astype(np.float64).values  # bins 8-bit a float64

    stats = summarize_histogram_matrix_robust(X, laplace_alpha=1e-9)

    b3 = three_letter_base(base)
    df_out[f"{b3}_{ABBR['mean']}"] = stats["mean"]
    df_out[f"{b3}_{ABBR['var']}"]  = stats["var"]
    df_out[f"{b3}_{ABBR['skew']}"] = stats["skew"]
    df_out[f"{b3}_{ABBR['kurt']}"] = stats["kurt"]


# 4.5 Ordenar: class primero si existe
if TARGET_COL in df_out.columns:
    cols_order = [TARGET_COL] + [c for c in df_out.columns if c != TARGET_COL]
    df_out = df_out[cols_order]

print("\nForma final:", df_out.shape)
print("Primeras columnas:", df_out.columns[:12].tolist())


/tmp/ipython-input-342683746.py:120: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[c] = pd.to_numeric(df[c], errors="ignore")


Histogramas detectados:
  ag: 10 bins -> ['ag_000', 'ag_001', 'ag_002', 'ag_003', 'ag_004'] ...
  ay: 10 bins -> ['ay_000', 'ay_001', 'ay_002', 'ay_003', 'ay_004'] ...
  az: 10 bins -> ['az_000', 'az_001', 'az_002', 'az_003', 'az_004'] ...
  ba: 10 bins -> ['ba_000', 'ba_001', 'ba_002', 'ba_003', 'ba_004'] ...
  cn: 10 bins -> ['cn_000', 'cn_001', 'cn_002', 'cn_003', 'cn_004'] ...
  cs: 10 bins -> ['cs_000', 'cs_001', 'cs_002', 'cs_003', 'cs_004'] ...
  ee: 10 bins -> ['ee_000', 'ee_001', 'ee_002', 'ee_003', 'ee_004'] ...

Columnas NO histogramas (muestra): ['clase', 'am_0', 'ec_00', 'aa_000', 'ab_000', 'ac_000', 'ad_000', 'ae_000', 'af_000', 'ah_000']


/tmp/ipython-input-342683746.py:93: RuntimeWarning: invalid value encountered in sqrt
  std = np.sqrt(var)
/tmp/ipython-input-342683746.py:93: RuntimeWarning: invalid value encountered in sqrt
  std = np.sqrt(var)
/tmp/ipython-input-342683746.py:93: RuntimeWarning: invalid value encountered in sqrt
  std = np.sqrt(var)
/tmp/ipython-input-342683746.py:93: RuntimeWarning: invalid value encountered in sqrt
  std = np.sqrt(var)
/tmp/ipython-input-342683746.py:93: RuntimeWarning: invalid value encountered in sqrt
  std = np.sqrt(var)
/tmp/ipython-input-342683746.py:93: RuntimeWarning: invalid value encountered in sqrt
  std = np.sqrt(var)



Forma final: (60000, 129)
Primeras columnas: ['clase', 'am_0', 'ec_00', 'aa_000', 'ab_000', 'ac_000', 'ad_000', 'ae_000', 'af_000', 'ah_000', 'ai_000', 'aj_000']


/tmp/ipython-input-342683746.py:93: RuntimeWarning: invalid value encountered in sqrt
  std = np.sqrt(var)


In [17]:
pd.set_option('display.max_columns', None)  # no cortar columnas
print(df_out.head())

      clase      am_0     ec_00    aa_000    ab_000    ac_000    ad_000  \
0 -0.992188 -0.109375  0.242188  0.117188 -0.289062  0.992188 -0.007812   
1 -0.992188 -0.109375  0.179688 -0.179688 -0.289062 -0.468750 -0.007812   
2 -0.992188 -0.109375 -0.132812 -0.125000 -0.289062 -0.468750 -0.007812   
3 -0.992188 -0.101562 -0.406250 -0.406250 -0.289062 -0.468750 -0.007812   
4 -0.992188 -0.109375 -0.109375  0.007812 -0.289062 -0.468750 -0.007812   

     ae_000    af_000    ah_000    ai_000    aj_000    ak_000    al_000  \
0 -0.046875 -0.054688  0.171875 -0.054688 -0.023438 -0.023438 -0.109375   
1 -0.046875 -0.054688 -0.101562 -0.054688 -0.023438 -0.023438 -0.109375   
2 -0.046875 -0.054688 -0.140625 -0.054688 -0.023438 -0.023438 -0.109375   
3 -0.046875 -0.007812 -0.429688 -0.054688 -0.023438 -0.023438 -0.109375   
4 -0.046875 -0.054688  0.039062 -0.054688 -0.015625 -0.023438 -0.109375   

     an_000    ao_000    ap_000    aq_000    ar_000    as_000    at_000  \
0  0.187500  0.093750  

In [21]:
import numpy as np
import pandas as pd

def audit_hist_group(df_out, cols, name="hist", top_n=5):
    X = df_out[cols].astype(float).values
    totals = np.nansum(X, axis=1)
    pos_bins = (X > 0).sum(axis=1)
    # var usando índices de bin como soporte, con PMF
    n_bins = X.shape[1]
    x = np.arange(n_bins).reshape(1, -1)
    totals_safe = np.where(totals==0, np.nan, totals)[:, None]
    p = X / totals_safe
    mu = np.nansum(p * x, axis=1)
    var = np.nansum(p * (x - mu[:, None])**2, axis=1)

    print(f"\n[AUDIT {name}] filas: {len(df)}")
    print(" total==0:", int((totals==0).sum()))
    print(" pos_bins==1 (toda la masa en un bin):", int((pos_bins==1).sum()))
    print(" var==0   :", int((var==0).sum()))
    # casos sospechosos: var==0 pero hay >1 bin positivo
    mask_weird = (var==0) & (pos_bins>1) & (totals>0)
    idxs = np.where(mask_weird)[0][:top_n]
    if len(idxs):
        print(f" casos sospechosos (var==0 & >1 bin>0): {len(idxs)} (mostrando {top_n}) ->", idxs.tolist())
        display(df.iloc[idxs][cols])
    else:
        print(" sin casos sospechosos (bien).")

# ejemplo: auditar todos los histogramas encontrados
for base, cols in hist_groups.items():
    audit_hist_group(df, cols, name=base)


[AUDIT ag] filas: 60000
 total==0: 66
 pos_bins==1 (toda la masa en un bin): 7818
 var==0   : 66
 sin casos sospechosos (bien).

[AUDIT ay] filas: 60000
 total==0: 90
 pos_bins==1 (toda la masa en un bin): 14252
 var==0   : 90
 sin casos sospechosos (bien).

[AUDIT az] filas: 60000
 total==0: 70
 pos_bins==1 (toda la masa en un bin): 13814
 var==0   : 70
 sin casos sospechosos (bien).

[AUDIT ba] filas: 60000
 total==0: 34
 pos_bins==1 (toda la masa en un bin): 9160
 var==0   : 34
 sin casos sospechosos (bien).

[AUDIT cn] filas: 60000
 total==0: 56
 pos_bins==1 (toda la masa en un bin): 7483
 var==0   : 56
 sin casos sospechosos (bien).

[AUDIT cs] filas: 60000
 total==0: 62
 pos_bins==1 (toda la masa en un bin): 6490
 var==0   : 62
 sin casos sospechosos (bien).

[AUDIT ee] filas: 60000
 total==0: 45
 pos_bins==1 (toda la masa en un bin): 6062
 var==0   : 45
 sin casos sospechosos (bien).


# 2. Preprocesamiento del dataset